# Dispositivos: CPU, GPU y dónde vive cada tensor

Una GPU puede entrenar una red muchas veces más rápido que la CPU, porque hace miles de operaciones en paralelo. Pero PyTorch no decide por su cuenta dónde se hace cada cálculo: cada tensor y cada modelo viven en un **dispositivo**, y moverlos es trabajo del programador. Cuando algo queda en el lugar equivocado aparece uno de los errores más comunes de PyTorch.

El módulo 1 lo mencionó de pasada (`00-fundamentos/01_tensores_autograd_y_una_neurona`, sección 8). Aquí vemos el patrón completo para escribir código que funcione igual en CPU y en GPU, cómo leer los errores de dispositivo y de qué depende la memoria que ocupa un entrenamiento.

**Sobre las salidas de este notebook.** Se ejecutó en un computador sin GPU disponible para PyTorch, así que todas las salidas dicen `cpu`. El código está escrito para usar la GPU si existe: en Colab se puede probar con *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*.

**Contenido**

1. Preparación
2. Todo tensor vive en un dispositivo
3. El error de dispositivos
4. `.to()` en tensores y en modelos
5. El patrón estándar
6. De vuelta a la CPU
7. Cuánta memoria ocupa un entrenamiento
8. Cuándo conviene la GPU
9. Resumen y ejercicios

## 1. Preparación

In [1]:
import time

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
print("Versión de PyTorch:", torch.__version__)

Versión de PyTorch: 2.11.0+cpu


## 2. Todo tensor vive en un dispositivo

Los nombres que usa PyTorch:

- **`cpu`**: el procesador y la memoria RAM del computador. Siempre está.
- **`cuda`**: una GPU de NVIDIA. Si hay varias se distinguen como `cuda:0`, `cuda:1`, y así.
- **`mps`**: la GPU de los Mac con procesador Apple (M1 en adelante).

La forma habitual de elegir es preguntar en orden y quedarse con lo mejor que haya:

In [2]:
def elegir_dispositivo():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


dispositivo = elegir_dispositivo()
print("cuda disponible:", torch.cuda.is_available())
print("mps disponible :", torch.backends.mps.is_available())
print("dispositivo elegido:", dispositivo)

cuda disponible: False
mps disponible : False
dispositivo elegido: cpu


Las versiones recientes de PyTorch traen además `torch.accelerator`, que responde lo mismo en una línea: `torch.accelerator.current_accelerator()` devuelve el acelerador disponible (`cuda`, `mps` u otro) o `None` si no hay ninguno.

Cada tensor sabe dónde vive, en su atributo `device`. Un modelo no tiene ese atributo, porque cada uno de sus parámetros podría estar en un lugar distinto. Se pregunta a uno de ellos:

In [3]:
x = torch.randn(3, 4)
modelo = nn.Linear(4, 2)

print("x.device                          :", x.device)
print("next(modelo.parameters()).device  :", next(modelo.parameters()).device)
print("acelerador:", torch.accelerator.current_accelerator())

x.device                          : cpu
next(modelo.parameters()).device  : cpu
acelerador: None


Los tensores se crean en la CPU, salvo que se pida otra cosa. Se puede crear un tensor directamente en un dispositivo con el argumento `device`, como en `torch.zeros(3, device=dispositivo)`.

## 3. El error de dispositivos

Una operación entre dos tensores necesita que los dos estén en el mismo dispositivo. PyTorch no mueve nada por su cuenta: si el modelo está en la GPU y el lote en la CPU, falla con un mensaje como este:

```
RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!
```

Para ver el error sin tener GPU usamos el dispositivo **`meta`**, que existe en cualquier computador. Un tensor en `meta` tiene forma y tipo, pero no tiene datos: sirve para calcular formas sin gastar memoria. Para PyTorch es un dispositivo distinto de la CPU, y eso basta:

In [4]:
modelo_en_meta = nn.Linear(4, 2).to("meta")
print("parámetros en:", next(modelo_en_meta.parameters()).device)

try:
    modelo_en_meta(x)                  # x sigue en la CPU
except RuntimeError as error:
    print("RuntimeError:", error)

parámetros en: meta
RuntimeError: Tensor on device meta is not on the expected device cpu!


El texto cambia según el dispositivo y la versión de PyTorch, pero siempre nombra los dos dispositivos que chocaron. Para encontrar al culpable, se imprime el `device` de cada cosa que participa en la operación: la entrada, las etiquetas, el modelo y la salida del modelo. Las etiquetas se olvidan seguido: el modelo y las imágenes están en la GPU, la predicción sale bien, y el error aparece recién en la pérdida.

In [5]:
def donde_esta(**cosas):
    for nombre, cosa in cosas.items():
        if isinstance(cosa, nn.Module):
            cosa = next(cosa.parameters())
        print(f"{nombre:10s} {cosa.device}")


donde_esta(entrada=x, modelo=modelo_en_meta, etiquetas=torch.tensor([0, 1, 1]))

entrada    cpu
modelo     meta
etiquetas  cpu


## 4. `.to()` en tensores y en modelos

`.to(dispositivo)` mueve algo a un dispositivo. Pero se comporta distinto según a qué se le aplique.

**En un tensor, `.to()` devuelve un tensor nuevo** y deja el original donde estaba. Si no se reasigna, no pasa nada:

In [6]:
x = torch.randn(3, 4)
x.to(torch.float64)                       # el resultado se pierde
print("sin reasignar:", x.dtype)

x = x.to(torch.float64)                   # reasignando
print("reasignando  :", x.dtype)

sin reasignar: torch.float32
reasignando  : torch.float64


Usamos un cambio de tipo porque en este computador no hay otro dispositivo al que mover, pero `.to()` es la misma función para las dos cosas: `x.to("cuda")` sin reasignar deja a `x` en la CPU, exactamente igual.

Un detalle: si el tensor ya está en el dispositivo y tipo pedidos, `.to()` devuelve el mismo objeto, sin copiar nada. Por eso `x = x.to(dispositivo)` es gratis cuando el dispositivo es la CPU, y el mismo código sirve en los dos casos.

In [7]:
y = torch.randn(3)
print("¿y.to('cpu') es el mismo objeto que y?             ", y.to("cpu") is y)
print("¿y.to(torch.float64) es el mismo objeto que y?     ", y.to(torch.float64) is y)

¿y.to('cpu') es el mismo objeto que y?              True
¿y.to(torch.float64) es el mismo objeto que y?      False


**En un modelo, `.to()` modifica el modelo en el lugar** y además lo devuelve. `modelo.to(dispositivo)` sin reasignar sí funciona:

In [8]:
modelo = nn.Linear(4, 2)
resultado = modelo.to(torch.float64)        # sin reasignar

print("¿Devolvió el mismo modelo?", resultado is modelo)
print("tipo de los pesos:", modelo.weight.dtype)

¿Devolvió el mismo modelo? True
tipo de los pesos: torch.float64


Esta diferencia es una fuente de errores: quien aprendió que `modelo.to(dispositivo)` funciona sin reasignar escribe `x.to(dispositivo)` sin reasignar, y el lote se queda en la CPU. La costumbre que evita el problema es reasignar siempre, también en el modelo: `modelo = modelo.to(dispositivo)`.

## 5. El patrón estándar

1. **Elegir el dispositivo una vez**, al principio.
2. **Mover el modelo una vez**, antes de crear el optimizador.
3. **Mover cada lote** dentro del ciclo, entradas y etiquetas.

El orden del paso 2 importa: el optimizador recibe los parámetros del modelo, y conviene que ya estén en su lugar. Así el estado interno del optimizador (los promedios de `Adam`, por ejemplo) se crea en el mismo dispositivo que los parámetros.

Las funciones de entrenamiento y evaluación reciben el dispositivo como argumento. Así el mismo código corre en CPU, en una GPU NVIDIA o en un Mac:

In [9]:
def entrenar_una_epoca(modelo, cargador, criterio, optimizador, dispositivo, max_lotes=None):
    modelo.train()
    for i, (x, y) in enumerate(cargador):
        if i == max_lotes:
            break
        x, y = x.to(dispositivo), y.to(dispositivo)      # 3. cada lote

        optimizador.zero_grad()
        perdida = criterio(modelo(x), y)
        perdida.backward()
        optimizador.step()
    return perdida.item()


def evaluar(modelo, cargador, dispositivo):
    modelo.eval()
    correctas = 0
    with torch.no_grad():
        for x, y in cargador:
            x, y = x.to(dispositivo), y.to(dispositivo)
            correctas += (modelo(x).argmax(dim=1) == y).sum().item()
    return correctas / len(cargador.dataset)


transformacion = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
entrenamiento = datasets.MNIST(root="../data", train=True, download=True, transform=transformacion)
prueba = datasets.MNIST(root="../data", train=False, download=True, transform=transformacion)
cargador_entrenamiento = DataLoader(entrenamiento, batch_size=64, shuffle=True)
cargador_prueba = DataLoader(prueba, batch_size=1000)

dispositivo = elegir_dispositivo()                                                  # 1. una vez
modelo = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
modelo = modelo.to(dispositivo)                                                     # 2. una vez...
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)                       #    ...antes del optimizador

perdida = entrenar_una_epoca(modelo, cargador_entrenamiento, nn.CrossEntropyLoss(), optimizador, dispositivo,
                             max_lotes=200)
print(f"dispositivo: {dispositivo}   pérdida del último lote: {perdida:.3f}   "
      f"exactitud de prueba tras 200 lotes: {evaluar(modelo, cargador_prueba, dispositivo):.1%}")

dispositivo: cpu   pérdida del último lote: 0.231   exactitud de prueba tras 200 lotes: 92.0%


Aquí entrenamos solo 200 lotes, para ver que el patrón funciona. El entrenamiento completo está en `04-clasificacion-de-imagenes`.

Hay dos ajustes que aceleran la copia de cada lote a la GPU, y que no hacen nada en la CPU: crear el `DataLoader` con `pin_memory=True`, que deja los lotes en una zona de la RAM desde la que la copia es más rápida, y mover con `x.to(dispositivo, non_blocking=True)`, que no detiene a la CPU mientras se hace la copia, así puede seguir preparando lo que viene.

## 6. De vuelta a la CPU

Algunas cosas solo funcionan con tensores en la CPU, sobre todo NumPy y matplotlib. Hay tres herramientas para traer resultados:

- **`.item()`**: convierte un tensor de un solo elemento en un número de Python, desde cualquier dispositivo. Es lo que se usa para anotar la pérdida.
- **`.cpu()`**: copia el tensor a la CPU. Es lo mismo que `.to("cpu")`.
- **`.detach()`**: separa el tensor del registro de autograd. NumPy no sabe de gradientes, así que no acepta tensores que los requieren.

In [10]:
pesos = modelo[1].weight                  # un parámetro: requiere gradientes
try:
    pesos.numpy()
except RuntimeError as error:
    print("RuntimeError:", error)

arreglo = pesos.detach().cpu().numpy()
print("con detach().cpu().numpy():", type(arreglo), arreglo.shape)

RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.
con detach().cpu().numpy(): <class 'numpy.ndarray'> (128, 784)


Con un tensor en la GPU, `.numpy()` sin `.cpu()` falla con un mensaje parecido: `can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.` La cadena `.detach().cpu().numpy()` funciona en los dos casos.

Cada copia entre la GPU y la CPU tiene un costo, y además obliga a esperar a que la GPU termine lo que estaba haciendo. Llamar `.item()` en cada paso para anotar la pérdida está bien; convertir lotes enteros a NumPy dentro del ciclo de entrenamiento lo vuelve lento.

## 7. Cuánta memoria ocupa un entrenamiento

La memoria de una GPU es más chica que la RAM de un computador: entre 4 y 32 GB en las tarjetas de uso personal. Cuando un entrenamiento no cabe, aparece `torch.OutOfMemoryError: CUDA out of memory`. Para saber de qué depende, separemos lo que se guarda durante el entrenamiento:

| Qué | Tamaño | Depende del lote |
|---|---|---|
| Pesos | 1 número por parámetro | no |
| Gradientes | 1 número por parámetro | no |
| Estado del optimizador | 0 en `SGD` (1 por parámetro con momento), 2 por parámetro en `Adam` | no |
| Lo que autograd retiene para `backward()` | según las capas | **sí, proporcional** |

Para la red de MNIST, con `Adam` y en `float32`:

In [11]:
parametros = sum(p.numel() for p in modelo.parameters())
print(f"parámetros: {parametros}")
print(f"pesos + gradientes + estado de Adam: 4 × {parametros} × 4 bytes = {4 * parametros * 4 / 1e6:.1f} MB")

parámetros: 101770
pesos + gradientes + estado de Adam: 4 × 101770 × 4 bytes = 1.6 MB


Unos pocos megabytes. Lo que crece es la cuarta fila. La medimos como en `01-modelos/01_nn_module` (sección 9), con una red algo más grande para que se note, y con varios tamaños de lote:

In [12]:
def memoria_retenida(modelo, entrada):
    pesos = {p.data_ptr() for p in modelo.parameters()}
    retenidos = {}

    def anotar(tensor):
        if tensor.data_ptr() not in pesos:
            retenidos[tensor.data_ptr()] = tensor.numel() * tensor.element_size()
        return tensor

    with torch.autograd.graph.saved_tensors_hooks(anotar, lambda tensor: tensor):
        modelo(entrada)
    return sum(retenidos.values())


red_grande = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 2048), nn.ReLU(),
    nn.Linear(2048, 2048), nn.ReLU(),
    nn.Linear(2048, 10),
)
fijo = 4 * sum(p.numel() for p in red_grande.parameters()) * 4
print(f"red 784 → 2048 → 2048 → 10: pesos, gradientes y Adam ocupan {fijo / 1e6:.0f} MB\n")

for lote in [64, 512, 4096, 8192]:
    retenido = memoria_retenida(red_grande, torch.randn(lote, 1, 28, 28))
    print(f"lote de {lote:4d}: retenido para backward {retenido / 1e6:6.1f} MB")

red 784 → 2048 → 2048 → 10: pesos, gradientes y Adam ocupan 93 MB

lote de   64: retenido para backward    1.2 MB
lote de  512: retenido para backward   10.0 MB


lote de 4096: retenido para backward   80.0 MB


lote de 8192: retenido para backward  159.9 MB


La memoria retenida crece en proporción directa al lote: de 64 a 512 ejemplos, 8 veces más memoria. Con lotes de 8.192 ya ocupa más que los pesos, los gradientes y el estado de `Adam` juntos. En redes convolucionales para fotos grandes la proporción es mucho más extrema, porque cada capa produce mapas de activación del tamaño de la imagen.

Por eso, cuando aparece el error de memoria, **lo primero es bajar el tamaño del lote**. Entre 32 y 64 es un buen punto de partida. Si hace falta un lote efectivo más grande, se acumulan gradientes de varios lotes chicos antes de cada `step()` (`02-entrenamiento/02_gradientes_y_optimizadores`, sección 7). Dos herramientas para mirar la memoria de la GPU desde el código: `torch.cuda.memory_allocated()` dice cuánta está ocupada ahora, y `torch.cuda.max_memory_allocated()` el máximo que se llegó a ocupar.

## 8. Cuándo conviene la GPU

Una GPU gana cuando hay mucho cálculo en paralelo: multiplicar matrices grandes, convolucionar imágenes. Medimos cuántas operaciones por segundo hace este computador multiplicando matrices cuadradas de distintos tamaños. Si hay GPU, se mide también ahí.

Medir tiempos en una GPU tiene una trampa: sus operaciones son **asíncronas**. La llamada vuelve enseguida y la GPU sigue calculando por detrás, así que hay que esperar con `torch.cuda.synchronize()` antes de mirar el reloj.

In [13]:
def medir_multiplicacion(n, dispositivo, repeticiones=10):
    a = torch.randn(n, n, device=dispositivo)
    b = torch.randn(n, n, device=dispositivo)
    a @ b                                                   # la primera vez tiene costos de arranque
    if dispositivo.type == "cuda":
        torch.cuda.synchronize()
    inicio = time.perf_counter()
    for _ in range(repeticiones):
        a @ b
    if dispositivo.type == "cuda":
        torch.cuda.synchronize()                            # esperar a que la GPU termine
    segundos = (time.perf_counter() - inicio) / repeticiones
    return segundos, 2 * n ** 3 / segundos / 1e9              # una multiplicación n × n hace 2n³ operaciones


dispositivos = [torch.device("cpu")] + ([torch.device("cuda")] if torch.cuda.is_available() else [])
for d in dispositivos:
    for n in [64, 256, 1024, 2048]:
        segundos, gflops = medir_multiplicacion(n, d)
        print(f"{d.type:4s} matrices de {n:4d} × {n:<4d}: {segundos * 1e3:8.3f} ms   {gflops:6.0f} miles de millones de operaciones por segundo")

cpu  matrices de   64 × 64  :    0.012 ms       45 miles de millones de operaciones por segundo
cpu  matrices de  256 × 256 :    0.225 ms      149 miles de millones de operaciones por segundo


cpu  matrices de 1024 × 1024:   14.938 ms      144 miles de millones de operaciones por segundo


cpu  matrices de 2048 × 2048:  107.358 ms      160 miles de millones de operaciones por segundo


En la CPU, las matrices chicas aprovechan mal el procesador: con 64 × 64 hay tan poco trabajo que los costos fijos de cada llamada pesan mucho. Desde 256 la velocidad se estabiliza en el máximo que da la CPU. Una GPU de gama media pasa de los diez billones de operaciones por segundo en `float32`, decenas de veces lo que da una CPU como esta, pero solo con matrices grandes: con matrices chicas no hay trabajo suficiente para ocupar sus miles de núcleos, y los costos fijos de cada llamada dominan.

Para la red 784 → 128 → 10 con lotes de 64, las matrices son chicas. En un entrenamiento así, buena parte del tiempo se va en cargar los datos (`00-datos`, sección 9) y la GPU ayuda poco. La diferencia de 10 a 15 veces que menciona el curso, o más, aparece con redes y lotes más grandes, sobre todo con redes convolucionales.

## 9. Resumen

- Cada tensor y cada modelo vive en un **dispositivo**: `cpu`, `cuda` (GPU NVIDIA) o `mps` (GPU de Apple). Se consulta con `tensor.device` y, en un modelo, con `next(modelo.parameters()).device`.
- PyTorch **no mueve nada solo**: operar entre dispositivos distintos da un `RuntimeError` que nombra los dos dispositivos. Hay que revisar entradas, etiquetas, modelo y salida.
- **`tensor.to()` devuelve un tensor nuevo** y hay que reasignarlo. **`modelo.to()` modifica el modelo** y lo devuelve. Reasignar siempre evita confundirse.
- El patrón: elegir el dispositivo al inicio, mover el modelo una vez antes de crear el optimizador y mover cada lote dentro del ciclo.
- Para volver a NumPy: `.detach().cpu().numpy()`. Para un número: `.item()`.
- La memoria de entrenar es pesos, gradientes y estado del optimizador (fijos), más lo que autograd retiene para `backward()`, **proporcional al lote**. Ante un error de memoria, primero se baja el lote.
- La GPU gana con matrices grandes. Para medirla hay que sincronizar, porque sus operaciones son asíncronas.

## Ejercicios

1. Escribe `elegir_dispositivo()` usando `torch.accelerator.current_accelerator()` y que devuelva la CPU cuando no haya acelerador.
2. Crea un modelo en `meta` y pásale un lote también en `meta`. ¿Funciona? ¿Qué forma tiene la salida? ¿Para qué podría servir calcular formas sin datos?
3. En `entrenar_una_epoca`, cambia `x, y = x.to(dispositivo), y.to(dispositivo)` por `x.to(dispositivo)` y `y.to(dispositivo)` sin reasignar. En una CPU no pasa nada. ¿Qué pasaría en una GPU, y en qué línea?
4. Mide con `memoria_retenida` la red 784 → 128 → 10 con lotes de 64 y de 1.000. ¿Qué capa aporta la mayor parte?
5. Si tienes acceso a una GPU (por ejemplo, en Colab), ejecuta la sección 8 allí. ¿Desde qué tamaño de matriz la GPU le gana a la CPU?
6. En Colab con GPU, entrena una época completa de MNIST con `entrenar_una_epoca` en CPU y en GPU, y mide el tiempo de cada una. ¿La diferencia se parece a la de la sección 8? ¿Por qué no?